# Stochastic Calculus, Step by Step · Lesson 9: The one-period binomial model

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **Part 2's teaching tree.** A share at S₀ = 100 that, every four months, goes up 10% (heads, H) or down 10% (tails, T); three steps make the course's one year. The real-world chance of heads is p = 0.6. The risk-free rate is the course's r = 3% (continuously compounded), so cash grows by R = e^0.01 per step, and the strike is K = 105. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. One step of the tree · 3. No arbitrage: d < R < u · 4. Copying the call: Δ shares and some cash · 5. Solving for the risk-neutral probability · 6. The real probability never appears · 7. Trading against a wrong price · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 9,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
import itertools

S0_, u, d, p = 100.0, 1.1, 0.9, 0.6          # the teaching tree: +10% (H) or -10% (T) per step, P(H) = 0.6
steps, dt_ = 3, 1 / 3                         # three steps of four months
R = math.exp(r * dt_)                         # cash growth per step
worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]   # HHH, HHT, ..., TTT


def price(w, n):
    """The share price after the first n tosses of world w."""
    return S0_ * u ** w[:n].count("H") * d ** w[:n].count("T")


def prob(w, q=p):
    """The probability of world w (or of the first tosses of one) when heads has probability q."""
    return q ** w.count("H") * (1 - q) ** w.count("T")


out["tree"] = {"S0": S0_, "u": u, "d": d, "p": p, "steps": steps, "dt": dt_, "R": R, "K": K}
pd.DataFrame({"world": worlds, "S1": [price(w, 1) for w in worlds], "S2": [price(w, 2) for w in worlds],
              "S3": [price(w, 3) for w in worlds], "P": [prob(w) for w in worlds]}).round(4)

## 2. One step of the tree

Take only the first step: four months. Three things can be traded today:

- the **share**, at 100 today, worth $S_0u = 110$ after heads or $S_0d = 90$ after tails;
- **cash**: 1 deposited today is worth $R = e^{0.03/3}$ in four months, whatever happens;
- a **call** with strike 105 that expires after this one step: it pays $\max(S_1 - 105, 0)$, so 5 after heads and 0 after tails.

What should the call cost today?

In [ ]:
Su, Sd = S0_ * u, S0_ * d
call_pay = (max(Su - K, 0.0), max(Sd - K, 0.0))
put_pay = (max(K - Su, 0.0), max(K - Sd, 0.0))
out["one"] = {"Su": Su, "Sd": Sd, "R": R, "R_pct": (R - 1) * 100, "Vu": call_pay[0], "Vd": call_pay[1],
              "Pu": put_pay[0], "Pd": put_pay[1], "K_disc": K / R}
out["one"]

## 3. No arbitrage: d < R < u

An **arbitrage** is a trade that costs nothing today, can never lose, and might win. If cash grew faster than the share in both outcomes ($R \ge u$), you would sell the share short and deposit the money; if the share always beat cash ($d \ge R$), you would borrow to buy it. A sensible market needs $d < R < u$. Ours has $0.9 < R < 1.1$.

In [ ]:
def short_and_deposit(R_):
    # Sell one share short, deposit the 100: the account minus the share you must give back, in each outcome.
    return {"after_H": S0_ * R_ - Su, "after_T": S0_ * R_ - Sd}


out["noarb"] = {"ok": d < R < u, "ours": short_and_deposit(R), "fast_cash": short_and_deposit(1.12)}
out["noarb"]

## 4. Copying the call: Δ shares and some cash

Hold $\Delta$ shares and $B$ in cash. After one step it is worth $\Delta S_0u + BR$ or $\Delta S_0d + BR$. Make both equal the call's payoff:

$$\Delta S_0u + BR = V_u, \qquad \Delta S_0d + BR = V_d .$$

Subtracting gives $\Delta = \dfrac{V_u - V_d}{S_0u - S_0d}$, and then $B = \dfrac{V_u - \Delta S_0u}{R}$. The copy costs $V_0 = \Delta S_0 + B$ today, and that must be the call's price.

In [ ]:
def replicate(Vu, Vd):
    Delta = (Vu - Vd) / (Su - Sd)
    B = (Vu - Delta * Su) / R
    return {"Delta": Delta, "B": B, "V0": Delta * S0_ + B, "share_cost": Delta * S0_,
            "end_H": Delta * Su + B * R, "end_T": Delta * Sd + B * R}


rep = replicate(*call_pay)
rep_put = replicate(*put_pay)
assert abs(rep["end_H"] - call_pay[0]) < 1e-12 and abs(rep["end_T"] - call_pay[1]) < 1e-12
out["rep"], out["rep_put"] = rep, rep_put
out["parity"] = {"lhs": rep["V0"] - rep_put["V0"], "rhs": S0_ - K / R}
pd.DataFrame({"call": rep, "put": rep_put}).round(4)

## 5. Solving for the risk-neutral probability

Put $\Delta$ and $B$ back into $V_0 = \Delta S_0 + B$ and collect the terms in $V_u$ and $V_d$:

$$V_0 = \frac{1}{R}\Big(\tilde p\,V_u + (1 - \tilde p)\,V_d\Big), \qquad \tilde p = \frac{R - d}{u - d} .$$

The price is a discounted average of the payoffs, with weights $\tilde p$ and $1 - \tilde p$: the **risk-neutral probabilities**. Under them the share earns exactly the cash rate: $\tilde pu + (1 - \tilde p)d = R$.

In [ ]:
pt = (R - d) / (u - d)
out["rn"] = {"pt": pt, "pt_pct": pt * 100, "one_minus_pt": 1 - pt, "V0": (pt * call_pay[0] + (1 - pt) * call_pay[1]) / R,
             "P0": (pt * put_pay[0] + (1 - pt) * put_pay[1]) / R, "growth": pt * u + (1 - pt) * d,
             "share_check": (pt * Su + (1 - pt) * Sd) / R}
assert abs(out["rn"]["V0"] - rep["V0"]) < 1e-12 and abs(out["rn"]["growth"] - R) < 1e-12
out["one"]["nodes"] = [{"t": 0, "k": 0, "price": S0_, "value": rep["V0"]},
                       {"t": 1, "k": 1, "price": Su, "value": call_pay[0]}, {"t": 1, "k": 0, "price": Sd, "value": call_pay[1]}]
out["rn"]

## 6. The real probability never appears

$p = 0.6$ was not used anywhere above. Whatever investors believe about heads, the copy costs the same, so the price is the same. The tempting answer, the real-world average payoff discounted, $p\,V_u/R$, depends on $p$ and is right only at $p = \tilde p$.

In [ ]:
ps = np.linspace(0, 1, 101)
out["only"] = {"p": (ps * 100).tolist(), "naive": (ps * call_pay[0] / R).tolist(), "flat": [rep["V0"]] * ps.size,
               "naive_at_p": p * call_pay[0] / R, "gap_at_p": p * call_pay[0] / R - rep["V0"],
               "ticks": [{"x": x, "label": f"{x}%"} for x in (0, 20, 40, 60, 80, 100)]}
{k_: out["only"][k_] for k_ in ("naive_at_p", "gap_at_p")}

## 7. Trading against a wrong price

If someone sells the call for less than the copy, buy the call and sell the copy (short $\Delta$ shares, deposit cash); if someone buys it for more, sell them the call and buy the copy. Either way the payoffs cancel at the end and the price gap is profit today.

In [ ]:
def arbitrage(market_price):
    if market_price > rep["V0"]:      # sell the call, buy the copy
        return {"action": "sell call, buy copy", "today": market_price - rep["V0"],
                "end_H": rep["end_H"] - call_pay[0], "end_T": rep["end_T"] - call_pay[1]}
    return {"action": "buy call, sell copy", "today": rep["V0"] - market_price,
            "end_H": call_pay[0] - rep["end_H"], "end_T": call_pay[1] - rep["end_T"]}


out["arb"] = {"high": {"price": 3.0, **arbitrage(3.0)}, "low": {"price": 2.5, **arbitrage(2.5)}}
out["arb_rows"] = [{"price": v["price"], "action": v["action"], "today": v["today"], "end_H": v["end_H"], "end_T": v["end_T"]}
                   for v in (out["arb"]["high"], out["arb"]["low"])]
pd.DataFrame(out["arb"]).T

## 8. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 2: Solutions*.

**Exercise 1.** Copy the put with strike 105 (it pays 0 after heads and 15 after tails): find $\Delta$, $B$ and the price. Why is $\Delta$ negative?

**Exercise 2.** Suppose cash grew by 12% per step instead ($R = 1.12$) while the share still moves to 110 or 90. Write down an arbitrage and its profit in each outcome. Do the same for $R = 0.88$.

**Exercise 3.** Solve the two copying equations for $\Delta$ and $B$ in general, substitute into $V_0 = \Delta S_0 + B$, and show that $V_0 = \big(\tilde pV_u + (1 - \tilde p)V_d\big)/R$ with $\tilde p = (R - d)/(u - d)$.

**Exercise 4.** Show that $\tilde p$ lies strictly between 0 and 1 exactly when $d < R < u$, and that $\big(\tilde pS_0u + (1 - \tilde p)S_0d\big)/R = S_0$.

**Exercise 5.** A digital option pays 10 if the first toss is heads and nothing otherwise. Find its price and its $\Delta$.

**Exercise 6.** A dealer quotes the call at 3.10. Describe the arbitrage trade step by step, and find the profit today and the position after each outcome.

**Exercise 7.** (Code) Write `one_period(S0, u, d, R, payoff)` that returns $\Delta$, $B$ and the price. Use it to check put-call parity $C - P = S_0 - K/R$ for strikes 92, 96, ..., 108.

**Exercise 8.** (Code) Plot the discounted real-world average payoff $pV_u/R$ and the no-arbitrage price as $p$ runs from 0 to 1. Where do they cross?

## 9. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")